# PaddleOCR-VL 보험사별 표 추출 시험 (Colab T4)

Drive `bench/.pdfs/`에 `<보험사>_<상품>.pdf`로 올린 문서들을 보험사별로 읽어(core 문서셋이 있으면 함께), **보험사마다** 괘선 표 2쪽·괘선 없는 표(gap) 2쪽·긴 표 1쪽·표 없는 본문 1쪽을 골라 qwen3-vl 8B(운영 기준선)와 PaddleOCR-VL을 같은 쪽에 돌린다. 마지막에 운영 모드(`VLM_TARGET=gap`)로 전체 문서를 Paddle로만 한 번 더 돌려 실제 투입 양과 시간을 본다.

전제
- 런타임 T4 GPU
- Drive `MyDrive/bench/.pdfs/`에 `메리츠_장기체류실손.pdf`처럼 보험사 접두어를 붙인 PDF를 올려 둔다. 접두어가 보험사 라벨이 된다
- 소요: 페이지 분류 5~8분, qwen 기준선 36쪽 30~50분, vLLM 설치 5~10분, Paddle 36쪽 2~3분, 운영 모드 전체 문서 10~20분


In [ ]:
# 1. Drive · 벤치 데이터 위치
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, sys, json, time, shutil, subprocess, hashlib, glob
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BENCH_DATA = '/content/drive/MyDrive/bench'
PDFS = Path(BENCH_DATA) / '.pdfs'
PDFS.mkdir(parents=True, exist_ok=True)
os.environ['BENCH_DATA'] = BENCH_DATA
print("BENCH_DATA =", BENCH_DATA)


In [ ]:
# 2. 코드 + 변환기 수정(가로 병합 셀 미반복)
%cd /content
!rm -rf Policy-Chunker && git clone -q https://github.com/SM17-YoonDongJu/Policy-Chunker
%cd /content/Policy-Chunker
!pip install -q -e .

# PaddleOCR-VL이 colspan으로 낸 가로 병합 셀을 변환기가 열 수만큼 반복해 문단이 4번 복제됐다(현대해상 안내 페이지).
# 세로 병합(행 헤더)은 그대로 반복하고 가로 병합은 첫 칸에만 쓴다. 같은 변경을 PR로 올릴 예정이라 여기서는 패치로 적용한다.
APPLY_COLSPAN_FIX = True
if APPLY_COLSPAN_FIX:
    p = Path('insurance_chunker/extractor.py'); src = p.read_text(encoding='utf-8')
    old1 = '''            for _ in range(n_col):
                out_row.append(text)
                if n_row > 1:
                    spans[col] = (text, n_row - 1)
                col += 1
'''
    new1 = '''            for k in range(n_col):
                cell = text if k == 0 else ""  # 가로 병합은 첫 칸에만 — 반복하면 문단이 열 수만큼 복제된다
                out_row.append(cell)
                if n_row > 1:
                    spans[col] = (cell, n_row - 1)
                col += 1
'''
    old2 = '''        elif tag == "lcel":
            row.append(row[-1] if row else "")
'''
    new2 = '''        elif tag == "lcel":
            row.append("")  # 왼쪽(가로) 병합: 반복하지 않는다
'''
    assert old1 in src and old2 in src, "extractor.py가 바뀌어 패치 대상이 없다 — 패치를 갱신해야 한다"
    p.write_text(src.replace(old1, new1).replace(old2, new2), encoding='utf-8')
    print("colspan 패치 적용")
!git diff --stat


In [ ]:
# 3. PDF 목록 — Drive bench/.pdfs/ 의 <보험사>_<상품>.pdf 를 보험사별 문서로 쓴다
#    예: 메리츠_장기체류실손.pdf → insurer=메리츠, 삼성화재_xxx.pdf → insurer=삼성화재
#    core 문서셋(sha256 이름)이 같이 있으면 그것도 포함한다. 그 밖의 파일(source.pdf, *-trial.pdf)은 무시한다
import yaml, re, pymupdf
DOCS = {}
for f in sorted(PDFS.glob('*_*.pdf')):
    insurer, _, product = f.stem.partition('_')
    did = re.sub(r'[^\w-]+', '-', f.stem).strip('-')        # 결과 파일명·yaml id. 한글 그대로 쓴다
    DOCS[did] = {'pdf': str(f), 'insurer': insurer, 'note': product, 'sha256': None}
core = yaml.safe_load(open('eval/bench/docsets/core.yaml', encoding='utf-8'))
for d in core['documents']:
    f = PDFS / f"{d['sha256']}.pdf"
    if f.exists():
        DOCS[d['id']] = {'pdf': str(f), 'insurer': d['insurer'], 'note': d.get('note', ''), 'sha256': d['sha256']}
assert DOCS, "문서가 없다 — Drive bench/.pdfs/ 에 <보험사>_<상품>.pdf 를 올린다"

print("사용할 문서:")
for k, v in DOCS.items():
    print(f"  {k:<28} {v['insurer']:<8} {os.path.getsize(v['pdf']) // 1024:>6}KB  {pymupdf.open(v['pdf']).page_count:>5}쪽  {v['note']}")
by_insurer = {}
for v in DOCS.values():
    by_insurer[v['insurer']] = by_insurer.get(v['insurer'], 0) + 1
print("보험사별 문서 수:", by_insurer)


In [ ]:
# 4. 문서마다 6쪽 선택 → <id>-trial.pdf  (괘선 표 2 · gap 2 · 긴 표 1 · 본문 1). 결과는 Drive에 캐시
import pymupdf
from insurance_chunker import extractor
SEL_DIR = Path(BENCH_DATA) / 'insurers-trial'; SEL_DIR.mkdir(exist_ok=True)
PNG_DIR = SEL_DIR / 'pages'; PNG_DIR.mkdir(exist_ok=True)

def pick_pages(pdf):
    doc = pymupdf.open(pdf); nums = list(range(1, doc.page_count + 1))
    sources, gap, _ = extractor.extract_cpu_tables(pdf, nums, use_vision=True)   # gap = PyMuPDF가 못 닿은 쪽
    pym = sources['pymupdf']
    by_len = sorted(pym, key=lambda p: -len(pym[p]))                      # 표 마크다운 길이 순
    plain = [p for p in nums if p not in pym and p not in gap]
    chosen = {}
    def take(kind, cands, n):
        for p in cands:
            if len([k for k in chosen.values() if k == kind]) >= n: break
            if p not in chosen: chosen[p] = kind
    take('long', by_len[:1], 1)
    take('ruled', [by_len[len(by_len) // 4], by_len[len(by_len) // 2]] if len(by_len) >= 4 else by_len, 2)
    take('gap', gap, 2)
    take('plain', plain[len(plain) // 2:], 1)
    take('ruled', by_len, 6 - len(chosen) + len([k for k in chosen.values() if k == 'ruled']))  # 모자라면 괘선 표로 채운다
    return dict(sorted(chosen.items())), {'pymupdf_pages': len(pym), 'gap_pages': len(gap), 'total': doc.page_count}

SEL = {}
for did, d in DOCS.items():
    trial = SEL_DIR / f'{did}-trial.pdf'; meta = SEL_DIR / f'{did}-pages.json'
    if trial.exists() and meta.exists():
        SEL[did] = json.load(open(meta)); print(f"{did}: 캐시 사용"); continue
    t0 = time.time()
    pages, info = pick_pages(d['pdf'])
    doc = pymupdf.open(d['pdf']); doc.select([p - 1 for p in pages]); doc.save(trial); doc.close()
    for i, page in enumerate(pymupdf.open(trial), 1):
        page.get_pixmap(dpi=110).save(PNG_DIR / f'{did}-p{i:02d}.png')
    SEL[did] = {'pages': {str(i): {'orig': p, 'kind': k} for i, (p, k) in enumerate(pages.items(), 1)}, **info}
    json.dump(SEL[did], open(meta, 'w'), ensure_ascii=False, indent=1)
    print(f"{did}: {info['total']}쪽 중 표 {info['pymupdf_pages']} / gap {info['gap_pages']} → 선택 {pages}  ({time.time() - t0:.0f}s)")


In [ ]:
# 5. 문서셋 · 변형 yaml
Path('eval/bench/docsets/insurers-trial.yaml').write_text(yaml.safe_dump({
    'name': 'insurers-trial',
    'documents': [{'id': did, 'pdf': str(SEL_DIR / f'{did}-trial.pdf'), 'type': d['insurer'],
                   'doc_type': 'policy_terms', 'insurer': d['insurer']} for did, d in DOCS.items()],
}, allow_unicode=True), encoding='utf-8')

# 운영 모드용 — 같은 문서들의 전체 PDF
Path('eval/bench/docsets/insurers-full.yaml').write_text(yaml.safe_dump({
    'name': 'insurers-full',
    'documents': [{'id': did, 'pdf': d['pdf'], 'type': d['insurer'],
                   'doc_type': 'policy_terms', 'insurer': d['insurer']} for did, d in DOCS.items()],
}, allow_unicode=True), encoding='utf-8')

COMMON = {'VLM_BACKEND': 'local', 'VLM_TARGET': 'both', 'VISION_MAX_PAGES': '6',
          'VLM_CONCURRENCY': '1', 'VLM_TIMEOUT': '600'}
PADDLE_SERVED = 'PaddleOCR-VL-1.5-0.9B'
VARIANTS = {
    'trial-qwen':   {'VLM_URL': 'http://localhost:11434', 'VLM_MODEL': 'qwen3-vl:8b-instruct'},
    'trial-paddle': {'VLM_URL': 'http://localhost:8000', 'VLM_MODEL': PADDLE_SERVED,
                     'VLM_PROMPT': 'Table Recognition:'},
    # 운영 모드: 전체 문서, gap 페이지만, 문서당 50쪽 상한 — Paddle만 돌린다(qwen은 시간이 몇 시간)
    'prod-paddle-gap': {'VLM_URL': 'http://localhost:8000', 'VLM_MODEL': PADDLE_SERVED,
                        'VLM_PROMPT': 'Table Recognition:', 'VLM_TARGET': 'gap', 'VISION_MAX_PAGES': '50'},
}
for slug, env in VARIANTS.items():
    Path(f'eval/bench/variants/{slug}.yaml').write_text(yaml.safe_dump(
        {'slug': slug, 'ref': 'WORKTREE', 'env': {**COMMON, **env}, 'status': 'draft'},
        allow_unicode=True), encoding='utf-8')

def show(slug, docset='insurers-trial'):
    print(f"[{slug}]")
    for did in DOCS:
        p = Path(BENCH_DATA) / 'results' / slug / docset / f'{did}.json'
        if not p.exists(): print(f"  {did:<24} 결과 없음"); continue
        r = json.load(open(p)); v = r['stages']['vlm']
        print(f"  {did:<24} 호출 {v.get('calls'):>3} 표 {v.get('tables'):>3}  p50 {v.get('latency_p50')}s  "
              f"p95 {v.get('latency_p95')}s  잘림 {v.get('truncated')}  {v.get('outcomes')}")
print("문서", len(DOCS), "건 ×", 6, "쪽")


In [ ]:
# 6. 기준선 qwen3-vl 8B (Ollama) — 36쪽 30~50분
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import requests
OLLAMA = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(OLLAMA), "ollama 설치 실패 — 위 로그 확인"
subprocess.Popen([OLLAMA, 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull qwen3-vl:8b-instruct
!grep -i "inference compute" /tmp/ollama.log     # library=cuda 여야 한다

!python eval/bench/bench.py run --docset insurers-trial --variant trial-qwen --to combine
!ollama ps
show('trial-qwen')
!ollama stop qwen3-vl:8b-instruct


In [ ]:
# 7. vLLM 설치 · PaddleOCR-VL 기동 (T4: --dtype half). Ollama 단계 뒤에 실행한다
!pip install -q uv
!uv pip install --system -q vllm --torch-backend=auto
!pip uninstall -y -q torchaudio          # Colab 기본 torchaudio가 교체된 torch와 CUDA 버전이 달라 import를 죽인다
!python -c "import torch, torchvision; print('torch', torch.__version__, torch.version.cuda, '/ torchvision', torchvision.__version__)"
!python -c "import vllm, transformers; print('vllm', vllm.__version__, '/ transformers', transformers.__version__)"

PADDLE_MODEL = 'PaddlePaddle/PaddleOCR-VL-1.5'
cmd = (f"vllm serve {PADDLE_MODEL} --trust-remote-code --dtype half --max-model-len 8192 "
       f"--max-num-batched-tokens 16384 --no-enable-prefix-caching --mm-processor-cache-gb 0 "
       f"--gpu-memory-utilization 0.5 --served-model-name {PADDLE_SERVED} --port 8000")
vllm_proc = subprocess.Popen(cmd.split(), stdout=open('/tmp/vllm.log', 'w'), stderr=subprocess.STDOUT)
ok = False
for _ in range(180):
    if vllm_proc.poll() is not None: break
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok: ok = True; break
    except Exception: pass
    time.sleep(5)
if not ok:
    print(open('/tmp/vllm.log').read()[-3000:]); raise SystemExit("vLLM이 뜨지 않았다 — 12번 llama.cpp 폴백으로")
print("vLLM 준비됨")


In [ ]:
# 8. PaddleOCR-VL 시험 — 36쪽 2~3분
!python eval/bench/bench.py run --docset insurers-trial --variant trial-paddle --to combine
show('trial-paddle')


In [ ]:
# 9. 보험사별 비교 — 요약표 + 페이지별 나란히 마크다운(Drive 저장)
import re
from IPython.display import display, Markdown, Image
from insurance_chunker.models import compute_doc_hash

def cells(mdtxt):
    out = set()
    for ln in (mdtxt or '').splitlines():
        if '|' in ln and not re.fullmatch(r'[\s|:-]+', ln):
            out |= {c.strip() for c in ln.strip().strip('|').split('|') if c.strip()}
    return out

def load_doc(did):
    trial = SEL_DIR / f'{did}-trial.pdf'
    ddir = Path(BENCH_DATA) / '.artifacts' / compute_doc_hash(str(trial))[:16]
    latest = lambda pat: max(glob.glob(str(ddir / pat)), key=os.path.getmtime)
    pym = json.load(open(latest('tables_cpu-*.json')))['output']['sources'].get('pymupdf', {})
    vl = {}
    for f in glob.glob(str(ddir / 'vlm-*.json')):
        d = json.load(open(f)); m = d.get('config', {}).get('model')
        if m: vl[m] = d['output']
    return pym, vl

MODELS = {'qwen3-vl:8b-instruct': 'qwen', PADDLE_SERVED: 'paddle'}
summary, lines = [], ['# 보험사별 VLM 표 추출 비교', '']
for did, d in DOCS.items():
    pym, vl = load_doc(did)
    sel = SEL[did]['pages']
    row = {'문서': did, '보험사': d['insurer']}
    for m, short in MODELS.items():
        out = vl.get(m, {})
        rep = Path(BENCH_DATA) / 'results' / f'trial-{short}' / 'insurers-trial' / f'{did}.json'
        v = json.load(open(rep))['stages']['vlm'] if rep.exists() else {}
        match = [len(cells(out.get(p)) & cells(pym[p])) / len(cells(pym[p]))
                 for p, info in sel.items() if info['kind'] in ('ruled', 'long') and p in pym and out.get(p) and cells(pym[p])]
        row[f'{short} 표'] = sum(1 for p in sel if out.get(p))
        row[f'{short} 셀일치'] = f"{sum(match) / len(match):.2f}" if match else '-'
        row[f'{short} 본문오탐'] = sum(1 for p, info in sel.items() if info['kind'] == 'plain' and out.get(p))
        row[f'{short} p50'] = v.get('latency_p50'); row[f'{short} 잘림'] = v.get('truncated')
    summary.append(row)
    lines += [f'## {did} ({d["insurer"]})', '']
    for p, info in sel.items():
        lines += [f'### p{p} — 원본 p{info["orig"]} [{info["kind"]}]', '', f'![{did}-p{p}](pages/{did}-p{int(p):02d}.png)', '',
                  '#### PyMuPDF', '', pym.get(p) or '(없음)', '']
        for m, short in MODELS.items():
            lines += [f'#### {short}', '', vl.get(m, {}).get(p) or '(출력 없음)', '']

import pandas as pd
df = pd.DataFrame(summary); display(df)
(SEL_DIR / 'compare.md').write_text('\n'.join(lines), encoding='utf-8')
print("페이지별 비교:", SEL_DIR / 'compare.md')

def show_doc(did):
    pym, vl = load_doc(did)
    for p, info in SEL[did]['pages'].items():
        display(Image(str(PNG_DIR / f'{did}-p{int(p):02d}.png'), width=480))
        for m, short in MODELS.items():
            display(Markdown(f"**p{p} [{info['kind']}] {short}**\n\n" + (vl.get(m, {}).get(p) or '(출력 없음)')))
# show_doc('kb-yes365-235p')   # 문서 하나를 눈으로 볼 때


In [ ]:
# 10. 운영 모드 — 올린 문서 전체를 gap 대상으로 Paddle만 (문서당 50쪽 상한). 보험사별 실제 투입 양·표 수·시간
!python eval/bench/bench.py run --docset insurers-full --variant prod-paddle-gap --to combine
for did, d in DOCS.items():
    p = Path(BENCH_DATA) / 'results' / 'prod-paddle-gap' / 'insurers-full' / f'{did}.json'
    if not p.exists(): print(f"  {did:<28} 결과 없음"); continue
    r = json.load(open(p)); s = r['stages']; v = s['vlm']; c = s['tables_cpu']
    print(f"  {did:<28} {d['insurer']:<6} 전체 {s['text'].get('pages')}쪽  PyMuPDF표 {c['table_pages'].get('pymupdf')}  gap {c.get('gap_pages')}  "
          f"VLM 호출 {v.get('calls')} 표 {v.get('tables')}  p50 {v.get('latency_p50')}s  VLM {v['seconds']}s / 전체 {r['elapsed_s']}s")


## 11. (선택) vLLM이 T4에서 안 뜨면 — llama.cpp 폴백
7번이 실패했을 때만. CUDA 빌드 10~15분. 이 경로로 돌린 결과는 모델명이 비어 있어 9번 셀의 `MODELS`에 `'': 'paddle'`을 추가해 읽는다.


In [ ]:
# 11. llama.cpp 빌드 · GGUF 서빙 · 벤치
try: vllm_proc.kill()
except Exception: pass
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON > /dev/null
!cmake --build /content/llama.cpp/build --config Release -j --target llama-server > /dev/null
llama_proc = subprocess.Popen(['/content/llama.cpp/build/bin/llama-server', '-hf', 'PaddlePaddle/PaddleOCR-VL-1.5-GGUF',
                               '--port', '8111', '--temp', '0', '-ngl', '99'],
                              stdout=open('/tmp/llama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get('http://localhost:8111/health', timeout=2).ok: break
    except Exception: time.sleep(5)
for slug in ('trial-paddle', 'prod-paddle-gap'):
    v = yaml.safe_load(open(f'eval/bench/variants/{slug}.yaml', encoding='utf-8'))
    v['env'].update({'VLM_URL': 'http://localhost:8111', 'VLM_MODEL': ''})
    Path(f'eval/bench/variants/{slug}-llama.yaml').write_text(yaml.safe_dump({**v, 'slug': f'{slug}-llama'}, allow_unicode=True), encoding='utf-8')
!python eval/bench/bench.py run --docset insurers-trial --variant trial-paddle-llama --to combine
show('trial-paddle-llama')


## 판정 기준

| 열 | 뜻 | 보는 법 |
|---|---|---|
| 표 | 6쪽 중 표를 낸 쪽 수 | ruled·long·gap 5쪽에서 나와야 한다 |
| 셀일치 | 괘선 표·긴 표에서 PyMuPDF 셀 대비 일치율 | 0.9 아래면 그 보험사 조판에서 깨지는 것. 숫자·코드 오독은 compare.md로 직접 확인 |
| 본문오탐 | 표 없는 쪽에서 표를 만든 수 | 0이어야 한다. Paddle은 태스크 토큰이라 1이 나올 수 있다 — 운영은 gap 대상이라 영향은 10번 셀 수치로 본다 |
| p50 · 잘림 | 페이지당 지연, max_tokens 잘림 | 운영 qwen p50 88.6초 대비 |

보험사별로 셀일치가 갈리면 그 보험사 조판(괘선 없는 표, 병합셀, 세로쓰기)에서 무엇이 깨지는지 compare.md의 해당 쪽을 본다. 결과는 Notion "모델 비교" 페이지에 보험사별 표로 옮긴다.
